# онтология сборка


In [4]:
!pip -q install pandas openpyxl requests beautifulsoup4 lxml

In [6]:
!pip install rdflib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 10.3 MB/s  0:00:00


In [7]:
import pandas as pd
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, OWL, XSD
import re

# ============================================================
# 1. НАСТРОЙКИ
# ============================================================
INPUT_CSV = "ontology_elder_speech_base_english.csv"
OUTPUT_TTL = "elder_speech_ontology.ttl"
BASE_URI = "http://example.org/elder-speech-ontology#"

In [8]:
# ============================================================
# 2. ЗАГРУЗКА + ДИАГНОСТИКА
# ============================================================
df = pd.read_csv(INPUT_CSV, encoding="utf-8-sig", dtype=str, keep_default_na=False)
print(f"Загружено строк: {len(df)}")
print(f"Колонки: {list(df.columns)}")

# Гибкая проверка названий колонок (на случай небольших расхождений)
def find_col(df, *candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

col_word = find_col(df, "Elder_Speech", "LexicalUnit", "Word")
col_mean = find_col(df, "Meaning", "Translation")
col_lang = find_col(df, "Real_Language", "Language", "NaturalLanguage")
col_word2 = find_col(df, "Corresponding_Word", "CorrespondingForm", "Parallel")
col_conf = find_col(df, "Confidence")
col_ref = find_col(df, "Reference", "Source_Note", "Note")
col_src = find_col(df, "Source")

assert col_word and col_mean and col_lang, \
    f"Не найдены обязательные колонки. Есть: {list(df.columns)}"

print(f"Используемые колонки: word={col_word}, meaning={col_mean}, "
      f"language={col_lang}, corresponding={col_word2}, "
      f"confidence={col_conf}, source={col_src}")

# Диагностика: сколько строк имеют непустой Real_Language
lang_series = df[col_lang].fillna("").astype(str).str.strip()
rows_with_lang = lang_series[(lang_series != "") & (lang_series.str.lower() != "nan")]
print(f"\n*** Строк с непустым Real_Language: {len(rows_with_lang)} из {len(df)}")
if len(rows_with_lang) > 0:
    print("Примеры значений:", rows_with_lang.head(10).tolist())
else:
    print("!!! ВНИМАНИЕ: колонка Real_Language пуста во всём файле.")
    print("!!! Проверьте, что вы загружаете правильный файл.")

# ============================================================
# 3. ФИКС PHERIAN (Sindarin → English)
# ============================================================
mask_pherian = df[col_word].str.casefold().str.strip() == "pherian"
if mask_pherian.any():
    print("\nИсправление pherian: Sindarin → English (concept)")
    df.loc[mask_pherian, col_lang] = "English"
    if col_word2:
        df.loc[mask_pherian, col_word2] = "halfling / hobbit"
    if col_ref:
        df.loc[mask_pherian, col_ref] = (
            "Literary borrowing from Tolkien's Sindarin 'perian'; "
            "conceptually corresponds to English 'halfling/hobbit'"
        )

Загружено строк: 224
Колонки: ['Elder_Speech', 'Meaning', 'Real_Language', 'Corresponding_Word', 'Confidence', 'Reference', 'Source', 'SourceType', 'SourceURL', 'Occurrences']
Используемые колонки: word=Elder_Speech, meaning=Meaning, language=Real_Language, corresponding=Corresponding_Word, confidence=Confidence, source=Source

*** Строк с непустым Real_Language: 49 из 224
Примеры значений: ['Dutch', 'Welsh', 'Italian', 'Welsh / Breton', 'Swedish', 'Irish', 'Irish', 'Irish', 'Irish', 'Welsh']

Исправление pherian: Sindarin → English (concept)


In [9]:
# ============================================================
# 4. ГРАФ + СХЕМА
# ============================================================
g = Graph()
EX = Namespace(BASE_URI)
g.bind("ex", EX)
g.bind("owl", OWL)
g.bind("rdfs", RDFS)

# Классы
for cls in [
    "FictionalLanguage", "LexicalUnit", "Meaning", "LexicalVariant",
    "NaturalLanguage", "Source", "LanguageVariant", "SpeakerCommunity",
    "WritingSystem", "CulturalConcept",
]:
    g.add((EX[cls], RDF.type, OWL.Class))

# Object Properties
obj_props = {
    "hasLexicalUnit":  ("FictionalLanguage", "LexicalUnit"),
    "hasMeaning":      ("LexicalUnit", "Meaning"),
    "hasVariant":      ("LexicalUnit", "LexicalVariant"),
    "influencedBy":    ("LexicalUnit", "NaturalLanguage"),
    "derivedFrom":     ("LexicalUnit", "NaturalLanguage"),
    "appearsIn":       ("LexicalUnit", "Source"),
    "spokenBy":        ("LanguageVariant", "SpeakerCommunity"),
    "hasWritingSystem":("NaturalLanguage", "WritingSystem"),
    "associatedWith":  ("LexicalUnit", "CulturalConcept"),
}
for prop, (domain, range_) in obj_props.items():
    g.add((EX[prop], RDF.type, OWL.ObjectProperty))
    g.add((EX[prop], RDFS.domain, EX[domain]))
    g.add((EX[prop], RDFS.range, EX[range_]))

# Datatype Properties (для хранения конкретного слова и уверенности)
for dp, rng in [
    ("hasCorrespondingForm", XSD.string),
    ("hasConfidence", XSD.string),
    ("hasReferenceNote", XSD.string),
]:
    g.add((EX[dp], RDF.type, OWL.DatatypeProperty))
    g.add((EX[dp], RDFS.domain, EX.LexicalUnit))
    g.add((EX[dp], RDFS.range, rng))

# ============================================================
# 5. ХЕЛПЕРЫ
# ============================================================
def make_uri(text: str) -> URIRef:
    safe = re.sub(r"[^\w\s-]", "", str(text)).strip().replace(" ", "_")
    return URIRef(BASE_URI + safe)

def split_languages(s: str) -> list:
    """Разбирает 'Welsh / Breton', 'Italian; French', 'English, Latin' и т.п."""
    if not s or not s.strip():
        return []
    parts = re.split(r"\s*[/;,&]\s*|\s+and\s+", s.strip())
    result = []
    for p in parts:
        p = p.strip()
        if not p:
            continue
        # Пропускаем вымышленные языки
        if p.casefold() in ("sindarin", "quenya", "klingon", "dothraki",
                            "hen linge", "elder speech"):
            print(f"  [пропуск вымышленного языка: {p}]")
            continue
        # Нормализация: 'welsh' → 'Welsh', 'irish / gaeilge' → 'Irish / Gaeilge' и т.п.
        if p.islower():
            p = p.title()
        result.append(p)
    # уникализируем, сохраняя порядок
    seen, out = set(), []
    for p in result:
        if p.casefold() not in seen:
            seen.add(p.casefold())
            out.append(p)
    return out

# ============================================================
# 6. ГЛАВНЫЙ ЯЗЫК
# ============================================================
elder_speech_uri = EX["ElderSpeech"]
g.add((elder_speech_uri, RDF.type, EX.FictionalLanguage))
g.add((elder_speech_uri, RDFS.label, Literal("Elder Speech", lang="en")))



<Graph identifier=N32dd46e4e8414d0ea708b19502a5a989 (<class 'rdflib.graph.Graph'>)>

In [10]:
# ============================================================
# 7. ЦИКЛ ПО СТРОКАМ
# ============================================================
stats = {
    "lexical_units": 0,
    "influencedBy": 0,
    "derivedFrom": 0,
    "languages": set(),
    "words_with_lang": [],
}
processed_units = set()  # чтобы не считать дубликаты в статистике

for _, row in df.iterrows():
    elder_word = str(row[col_word]).strip()
    if not elder_word:
        continue

    lu_uri = make_uri(f"lu_{elder_word}")
    g.add((lu_uri, RDF.type, EX.LexicalUnit))
    g.add((lu_uri, RDFS.label, Literal(elder_word, lang="en")))
    g.add((elder_speech_uri, EX.hasLexicalUnit, lu_uri))

    if elder_word not in processed_units:
        stats["lexical_units"] += 1
        processed_units.add(elder_word)

    # --- Meaning ---
    meaning = str(row[col_mean]).strip()
    if meaning:
        meaning_uri = make_uri(f"meaning_{elder_word}")
        g.add((meaning_uri, RDF.type, EX.Meaning))
        g.add((meaning_uri, RDFS.label, Literal(meaning, lang="en")))
        g.add((lu_uri, EX.hasMeaning, meaning_uri))

    # --- Real Language (главное!) ---
    real_lang_raw = str(row[col_lang]).strip() if col_lang else ""
    confidence = str(row[col_conf]).strip().lower() if col_conf else ""
    corresponding = str(row[col_word2]).strip() if col_word2 else ""

    if real_lang_raw and real_lang_raw.lower() not in ("", "nan", "none"):
        languages = split_languages(real_lang_raw)
        for lang_name in languages:
            lang_uri = make_uri(f"lang_{lang_name}")
            g.add((lang_uri, RDF.type, EX.NaturalLanguage))
            g.add((lang_uri, RDFS.label, Literal(lang_name, lang="en")))

            # ВСЕГДА добавляем influencedBy
            g.add((lu_uri, EX.influencedBy, lang_uri))
            stats["influencedBy"] += 1
            stats["languages"].add(lang_name)

            # derivedFrom — только при высокой уверенности
            if confidence == "high":
                g.add((lu_uri, EX.derivedFrom, lang_uri))
                stats["derivedFrom"] += 1

            if elder_word not in [w for w, _, _ in stats["words_with_lang"]]:
                stats["words_with_lang"].append((elder_word, lang_name, confidence))

    # --- Конкретное соответствующее слово ---
    if corresponding:
        g.add((lu_uri, EX.hasCorrespondingForm, Literal(corresponding)))
    if confidence:
        g.add((lu_uri, EX.hasConfidence, Literal(confidence)))

    # --- Reference ---
    ref = str(row[col_ref]).strip() if col_ref else ""
    if ref:
        g.add((lu_uri, EX.hasReferenceNote, Literal(ref)))

    # --- Sources ---
    sources = str(row[col_src]).strip() if col_src else ""
    if sources:
        for s_name in re.split(r"\s*\|\s*", sources):
            s_name = s_name.strip()
            if s_name:
                s_uri = make_uri(f"source_{s_name}")
                g.add((s_uri, RDF.type, EX.Source))
                g.add((s_uri, RDFS.label, Literal(s_name, lang="en")))
                g.add((lu_uri, EX.appearsIn, s_uri))

# ============================================================
# 8. СОХРАНЕНИЕ
# ============================================================
g.serialize(destination=OUTPUT_TTL, format="turtle")
print(f"\nСохранено: {OUTPUT_TTL}")
print(f"Всего триплетов в памяти: {len(g)}")

# ============================================================
# 9. ПРОВЕРКА: перечитываем файл и считаем influencedBy
# ============================================================
print("\n" + "="*60)
print("ПРОВЕРКА ФАЙЛА (перечитываем TTL)")
print("="*60)

check_g = Graph()
check_g.parse(OUTPUT_TTL, format="turtle")

influenced_triples = list(check_g.triples((None, EX.influencedBy, None)))
derived_triples = list(check_g.triples((None, EX.derivedFrom, None)))
natural_langs = list(check_g.triples((None, RDF.type, EX.NaturalLanguage)))

print(f"influencedBy триплетов в файле: {len(influenced_triples)}")
print(f"derivedFrom триплетов в файле: {len(derived_triples)}")
print(f"NaturalLanguage индивидов: {len(natural_langs)}")

if len(influenced_triples) == 0:
    print("\n!!! КРИТИЧЕСКАЯ ОШИБКА: influencedBy триплетов нет!")
    print("Причина: колонка Real_Language пуста во всех строках.")
    print(f"Пример первых 5 значений: {lang_series.head().tolist()}")
    print("Проверьте, что загружаете ontology_elder_speech_base_english.csv,")
    print("а не elder_speech_all.csv или другой промежуточный файл.")
else:
    print(f"\n✓ УСПЕХ: {len(influenced_triples)} связей influencedBy сохранены в файл")
    print(f"✓ {len(natural_langs)} уникальных NaturalLanguage индивидов")
    print(f"\nПримеры (первые 10):")
    for s, p, o in influenced_triples[:10]:
        lu_label = str(s).split("#")[-1].replace("lu_", "")
        lang_label = str(o).split("#")[-1].replace("lang_", "")
        print(f"  {lu_label} → influencedBy → {lang_label}")

# ============================================================
# 10. СПРАВКА ДЛЯ PROTEGE
# ============================================================
print("\n" + "="*60)
print("ЧТО ДЕЛАТЬ В PROTEGE")
print("="*60)
print("""
1. File → Open → выберите elder_speech_ontology.ttl

2. Вкладка 'Individuals by class' → выберите LexicalUnit →
   вы увидите все лексические единицы (aep, caer, beag, ...).

3. Кликните на любую (например, lu_aep) — справа в 'Object property
   assertions' должна быть строка:
       influencedBy  lang_Welsh
       derivedFrom   lang_Welsh

4. Вкладка 'Object properties' → influencedBy → внизу 'Property assertions'
   покажет все связи LexicalUnit → NaturalLanguage.

5. Если хотите SPARQL-проверку: вкладка 'SPARQL Query', введите:

    PREFIX ex: <http://example.org/elder-speech-ontology#>
    SELECT ?word ?lang WHERE {
      ?word ex:influencedBy ?lang .
    } LIMIT 20

   Должно вернуть строки (aep, Welsh), (caer, Welsh) и т.д.
""")


Сохранено: elder_speech_ontology.ttl
Всего триплетов в памяти: 2321

ПРОВЕРКА ФАЙЛА (перечитываем TTL)
influencedBy триплетов в файле: 56
derivedFrom триплетов в файле: 41
NaturalLanguage индивидов: 12

✓ УСПЕХ: 56 связей influencedBy сохранены в файл
✓ 12 уникальных NaturalLanguage индивидов

Примеры (первые 10):
  Bloed → influencedBy → Dutch
  Enid → influencedBy → Welsh
  aep → influencedBy → Welsh
  aevon → influencedBy → Welsh
  bleid → influencedBy → Welsh
  caer → influencedBy → Welsh
  fraeren → influencedBy → Welsh
  glaeddyv → influencedBy → Welsh
  gleidyf → influencedBy → Welsh
  gweld → influencedBy → Welsh

ЧТО ДЕЛАТЬ В PROTEGE

1. File → Open → выберите elder_speech_ontology.ttl

2. Вкладка 'Individuals by class' → выберите LexicalUnit →
   вы увидите все лексические единицы (aep, caer, beag, ...).

3. Кликните на любую (например, lu_aep) — справа в 'Object property
   assertions' должна быть строка:
       influencedBy  lang_Welsh
       derivedFrom   lang_Welsh

4. В